In [27]:
from dotenv import load_dotenv
import os
import kagglehub
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix, ConfusionMatrixDisplay
from TabTransformer import run_experiment as run_experiment_c, infer_tabtransformer as infer_tabtransformer_c
from qCLSTabTransformer import run_experiment as run_experiment_q, infer_tabtransformer as infer_tabtransformer_q
from utils import preprocess_amex, checkpoint_from_model, model_from_checkpoint
from qiskit_ibm_runtime import QiskitRuntimeService
import pennylane as qml
from pennylane_qiskit import qiskit_session

%load_ext autoreload
%autoreload 2

load_dotenv()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [ ]:
# #Run once to connect IBM quantum platform account
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token=os.environ["IBM_API_KEY"],
#     instance=os.environ["IBM_INSTANCE_CRN"],
#     set_as_default=True,
#     overwrite=True,
# )

In [3]:
#Use .env file with Kaggle API Token to download AMEX dataset (Don't run this if you want to use the UGB dataset)
load_dotenv()
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")
path = kagglehub.competition_download("amex-default-prediction")
file = os.path.join(path, "train_data.csv")
data_all = pd.read_csv(file, nrows=1e6)
file = os.path.join(path, "train_labels.csv")
labels_all = pd.read_csv(file,nrows=1e6)

#Scrape all data associated with the a random n customers
customers = data_all["customer_ID"].drop_duplicates().sample(n=12000, random_state=42)
data_without_labels = data_all[data_all["customer_ID"].isin(customers)]
labels = labels_all[labels_all["customer_ID"].isin(customers)]
data = preprocess_amex(data_without_labels, labels)
frac_fraud = np.sum(data['Class'])/len(data['Class'])

print(f"Size of dataset: {len(data)}")
print(f"Number of features: {data.shape[1]}")
print(f"# of Frauds = {int(np.sum(data['Class']))}")
print(frac_fraud*100, '% fraud')

Size of dataset: 12000
Number of features: 226
# of Frauds = 3124
26.033333 % fraud


In [28]:
#Split data into fraud and non-fraud to build high concentration of fraud in small training dataset
fraud_data = data[data['Class'] == 1].sample(frac=1, random_state=42)
non_fraud_data = data[data['Class'] == 0].sample(frac=1, random_state=42)

train_size = 2000

#UGB data split uses all available fraud in main experiment, while AMEX keeps same fraud ratio

#data_subset = pd.concat([fraud_data, non_fraud_data[:train_size - len(fraud_data)]]).sample(frac=1, random_state=42) #Use for MLG-UGB
training_fraud = int(frac_fraud*train_size) #Use for AMEX
data_subset = pd.concat([fraud_data[:training_fraud], non_fraud_data[:train_size - training_fraud]]).sample(frac=1, random_state=42) #Use for AMEX

#Run classical experiment on small dataset
model_c, test_results_c, test_df = run_experiment_c(data_subset, lr=3e-3)
torch.save(checkpoint_from_model(model_c), "classical_model_test.pt")
print(f'Classical model test AUPRC: {test_results_c["auprc"]}')

#Run quantum experiment on small dataset
model_q, test_results_q, test_df = run_experiment_q(data_subset, lr=3e-3, qufex_params=(4,1,1))
torch.save(checkpoint_from_model(model_q), "quantum_model_test.pt")
print(f'Quantum CLS model test AUPRC: {test_results_q["auprc"]}')

Classical model test AUPRC: 0.8590841676625717
Quantum CLS model test AUPRC: 0.8834955315916441


In [29]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=4,
)

print("Selected backend:", backend.name)
print("Backend qubits:", backend.num_qubits)

Selected backend: ibm_fez
Backend qubits: 156


In [30]:
model = model_from_checkpoint(torch.load("quantum_model_test.pt", map_location="cpu", weights_only=True))

In [31]:
size = 5
inference_fraud = int(frac_fraud * size)
inference_df = pd.concat([fraud_data[training_fraud:training_fraud + inference_fraud],
                            non_fraud_data[train_size - training_fraud:train_size - training_fraud + size - inference_fraud]]
                            ).sample(frac=1, random_state=42) #Use for AMEX dataset

results = infer_tabtransformer_q(model, inference_df)
print(f'AUPRC:          {results["auprc"]}')
print(f'Fraud Scores:   {results["probs"]}')
print(f'Predictions:    {results["preds"]}')
print(f'Targets:        {results["targets"]}')

AUPRC:          0.5
Fraud Scores:   [0.17661165 0.1664504  0.59977174 0.5657849  0.16095397]
Predictions:    [0 0 1 1 0]
Targets:        [0. 0. 0. 1. 0.]


In [32]:
q_device = qml.device(
    "qiskit.remote",
    wires=backend.num_qubits,
    backend=backend,
    resilience_level=1,
    optimization_level=3,
    seed_transpiler=42,
)

with qiskit_session(q_device, max_time="5m") as session:
    results = infer_tabtransformer_q(model, inference_df, quantum_device=q_device, shots=100)

print(f'AUPRC:          {results["auprc"]}')
print(f'Fraud Scores:   {results["probs"]}')
print(f'Predictions:    {results["preds"]}')
print(f'Targets:        {results["targets"]}')

AUPRC:          0.5
Fraud Scores:   [0.17606467 0.16644816 0.59795254 0.5647898  0.16151445]
Predictions:    [0 0 1 1 0]
Targets:        [0. 0. 0. 1. 0.]
